In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ഘട്ടം 1: ഘടനാപരമായ ഔട്ട്പുട്ടുകൾക്കായി Pydantic മോഡലുകൾ നിർവചിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ നൽകുന്ന **സ്കീമ** നിർവചിക്കുന്നു. Pydantic-നെ ഉപയോഗിച്ച് `response_format` ചെയ്യുമ്പോൾ ഉറപ്പാക്കുന്നത്:
- ✅ തര-സുരക്ഷിത ഡാറ്റാ പുറത്തെടുക്കൽ
- ✅ സ്വയം-പരിശോധന
- ✅ സ്വതന്ത്ര-വാചക പ്രതികരണങ്ങളിൽ നിന്നും പാഴ്‌സിംഗ് പിശകുകൾ വടക്കാതെ
- ✅ ഫീൽഡുകൾ ആസ്പദമാക്കി എളുപ്പത്തിൽ നിബന്ധനാപരമായ റൂട്ടിംഗ്


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ഘട്ടം 2: ഹോട്ടൽ ബുക്കിംഗ് ടൂൾ സൃഷ്ടിക്കുക

ഈ ടൂൾ **availability_agent** റൂമുകൾ ലഭ്യമായിരിക്കുന്നുവോ എന്ന് പരിശോധിക്കാൻ വിളിക്കും. `@ai_function` ഡെക്കറേറ്റർ ഞങ്ങൾ താഴെ പറയുന്നതിനായി ഉപയോഗിക്കുന്നു:
- ഒരു Python ഫങ്ഷനെ AI-കോളബിള് ടൂളാക്കി മാറ്റുക
- LLM ന് സ്വയം JSON സ്‌കീമ സൃഷ്ടിക്കുക
- പാരാമീറ്റർ സാധുത പരിശോധന കൈകാര്യം ചെയ്യുക
- ഏജൻ്റ്സുകൾക്കും സ്വയം പ്രയോഗം നടത്താൻ കഴിയുക

ഈ ഡെമോയ്ക്ക്:
- **സ്റ്റോക്ക്ഹോം, സിയാറ്റിൽ, ടോക്കിയോ, ലണ്ടൻ, ആംസ്റ്റർഡാം** → റൂമുകൾ ലഭ്യമാണ് ✅
- **മറ്റു എല്ലാ നഗരങ്ങളും** → റൂമുകൾ ഇല്ല ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ഘട്ടം 3: റൂട്ടിംഗിനായി നില ഫังก്ഷനുകൾ定義ചെയ്‌യുക

ഈ ഫังก്ഷനുകൾ ഏജന്റിന്റെ മറുപടി പരിശോധിച്ച് വർക്ക്ഫ്ലോയിൽ ഏത് പാത പിന്തുടരണമെന്ന് നിർണ്ണയിക്കും.

**പ്രധാന പരിണതി:**
1. സന്ദേശം `AgentExecutorResponse` ആണോ എന്ന് പരിശോധിക്കുക
2. ഘടനയുള്ള ഔട്ട്പുട്ട് (Pydantic മോഡൽ) പാഴ്‌സുചെയ്യുക
3. റൂട്ടിംഗ് നിയന്ത്രിക്കാൻ `True` അല്ലെങ്കിൽ `False` തിരിച്ചു നൽകുക

വർക്ക്ഫ്ലോ ഈ നിലകൾ **എഡ്ജുകളിൽ** വിലയിരുത്തി അടുത്ത എക്‌സിക്യൂട്ടർ ആരെന്ന് തീരുമാനിക്കും.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## പടി 4: കസ്റ്റം ഡിസ്‌പ്ലേ Executes സൃഷ്ടിക്കുക

Executes മൊഴിമാറ്റങ്ങൾക്ക് അല്ലെങ്കിൽ saide ഫലങ്ങൾക്ക് പ്രവർത്തിക്കുന്ന വർക്ക്‌ഫ്ലോ ഘടകങ്ങളാണ്. അവസാന ഫലം പ്രദർശിപ്പിക്കുന്ന കസ്റ്റം Executes സൃഷ്ടിക്കാൻ `@executor` ഡെക്കറേറ്റർ ഉപയോഗിക്കുന്നു.

**പ്രധാന ആശയങ്ങൾ:**
- `@executor(id="...")` - ഫംഗ്ഷൻ ഒരു വർക്‌ഫ്ലോ Executes ആയി രജിസ്റ്റർ ചെയ്യുക
- `WorkflowContext[Never, str]` - ഇൻപുട്ട്/ഔട്ട്പുട്ടിന് ടൈപിൻറുകൾ
- `ctx.yield_output(...)` - അന്തിമ വർക്‌ഫ്ലോ ഫലം അയയ്ക്കുക


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## പടി 5: പരിസ്ഥിതി ഘടകങ്ങൾ ലോഡ് ചെയ്യുക

LLM ക്ലയറന്റ് കോൺഫിഗർ ചെയ്യുക. ഈ ഉദാഹരണം പ്രവർത്തിക്കുന്നു:
- **Microsoft Foundry** / **Azure OpenAI** (പ്രതികരണങ്ങൾ API — ശുപാർശ ചെയ്യപ്പെടുന്നു)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ഘട്ടം 6: ഘടനാപരമായ ഔട്ട്‌പുട്ടുകളോടെ AI ഏജൻറുകൾ സൃഷ്ടിക്കുക

നാം **മൂന്നു പ്രത്യേക ഏജൻറുകളോരോവണ്ണം** സൃഷ്ടിക്കുന്നു, ഓരോന്നും `AgentExecutor` ൽ ചേർക്കപ്പെട്ടിരിക്കുന്നു:

1. **availability_agent** - ഹോട്ടൽ ലഭ്യത ടൂളിന്റെ സഹായത്തോടെ പരിശോധിക്കുന്നു
2. **alternative_agent** - അലേർണേറ്റിവ് പട്ടണങ്ങൾ നിർദ്ദേശിക്കുന്നു (മുറികൾ ഇല്ലാത്തപ്പോൾ)
3. **booking_agent** - ബുക്കിങ് പ്രോത്സാഹിപ്പിക്കുന്നു (മുറികൾ ലഭ്യമായപ്പോൾ)

**പ്രധാന സവിശേഷതകൾ:**
- `tools=[hotel_booking]` - ഏജന്റിന് ടൂൾ നൽകുന്നു
- `response_format=PydanticModel` - ഘടനാപരമായ JSON ഔട്ട്പുട്ട് നിർബന്ധിക്കുന്നു
- `AgentExecutor(..., id="...")` - വർക്ക്‌ഫ്ലോ ഉപയോഗത്തിന് ഏജന്റ് റാപ്പ് ചെയ്യുന്നു


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ഘട്ടം 7: നിബന്ധനാപരമായ അറ്റകുറ്റപ്പണി ഉപയോഗിച്ചു പ്രവാഹം നിർമ്മിക്കുക

ഇപ്പോൾ നമുക്ക് `WorkflowBuilder` ഉപയോഗിച്ച് നിബന്ധനാപരമായ റൂട്ടിംഗ് ഉള്ള ഗ്രാഫ് നിർമ്മിക്കാം:

**പ്രവാഹഘടന:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**പ്രധാന കാർയ്യങ്ങൾ:**
- `.set_start_executor(...)` - പ്രവേശനാദ്ധാരംക്രമം സജ്ജമാക്കുന്നു
- `.add_edge(from, to, condition=...)` - നിബന്ധനാപരമായ അറ്റകുറ്റപ്പണി ചേർക്കുന്നു
- `.build()` - പ്രവാഹം അവസാനിപ്പിക്കുന്നു


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ഘട്ടം 8: ടെസ്റ്റ് കേസ് 1 ഓടിക്കുക - ലഭ്യതയില്ലാത്ത നഗരം (പാരിസ്)

നമ്മുടെ സിമുലേഷനിൽ മുറികൾ ഇല്ലാത്ത പാരിസിൽ ഹോട്ടലുകൾ ആവശ്യപ്പെട്ട് **ലഭ്യത ഇല്ലാത്ത** മാർഗം പരിശോധിക്കാം.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ഘട്ടം 9: ടെസ്റ്റ് കേസ്സ് 2 - ലഭ്യതയോടുകൂടെ നഗരത്തിൽ പ്രവർത്തിപ്പിക്കുക (സ്റ്റോക്ക്‌ഹോൾം)

ഇനി ഞങ്ങൾ **ലഭ്യത** പാത പരിശോധിക്കാം സ്റ്റോക്ക്‌ഹോൾമിലെ (നമ്മുടെ സിമുലേഷനിൽ മുറികളുള്ള) ഹോട്ടലുകൾ തേടി.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## പ്രധാനപ്പെട്ട കാര്യങ്ങളും അടുത്ത ചുവടുകളും

### ✅ നിങ്ങൾ പഠിച്ച കാര്യങ്ങൾ:

1. **WorkflowBuilder പാറ്റേൺ**
   - എൻട്രി പോയിന്റ് നിർവ്വചിക്കാൻ `.set_start_executor()` ഉപയോഗിക്കുക
   - നിബന്ധനാപூரിത റൂട്ടിംഗിന് `.add_edge(from, to, condition=...)` ഉപയോഗിക്കുക
   - വർക്ക്‌ഫ്ലോ ഫൈനലൈസുചെയ്യാൻ `.build()` വിളിക്കാം

2. **നിബന്ധനാപൂർണ റൂട്ടിംഗ്**
   - കൺഡീഷൻ ഫങ്ഷനുകൾ `AgentExecutorResponse` പരിശോധിക്കുന്നു
   - റൂട്ടിംഗ് തീരുമാനങ്ങൾ എടുക്കാൻ ഘടിതഔട്ട്പുട്ടുകൾ പാർസ് ചെയ്യുക
   - ഒരു എഡ്ജ് സജീവമാക്കാൻ `True` മടക്കുക, അത് ഒഴിവാക്കാൻ `False`

3. **ഉപകരണ ഇന്റഗ്രേഷൻ**
   - Python ഫങ്ഷനുകൾ എ ഐ ഉപകരണങ്ങളാക്കി മാറ്റാൻ `@ai_function` ഉപയോഗിക്കുക
   - ആവശ്യപ്പെടുമ്പോൾ ഏജൻറുകൾ സ്വയം ഉപകരണങ്ങൾ വിളിക്കുന്നു
   - ഉപകരണങ്ങൾ JSON മടക്കി നൽകുന്നു, ഏജൻറുകൾ അത് പാർസ് ചെയ്യുന്നു

4. **ഘടിത ഔട്ട്പുട്ടുകൾ**
   - ടൈപ്പ്-സുരക്ഷിത ഡാറ്റ എക്സ്ട്രാക്ഷനായി Pydantic മോഡലുകൾ ഉപയോഗിക്കുക
   - ഏജൻറുകൾ സൃഷ്ടിക്കുമ്പോൾ `response_format=MyModel` സജ്ജമാക്കുക
   - `Model.model_validate_json()` ഉപയോഗിച്ച് റസ്പോൺസുകൾ പാർസ് ചെയ്യുക

5. **സ്വതന്ത്ര എക്സിക്യൂട്ടറുകൾ**
   - വർക്ക്‌ഫ്ലോ ഘടകങ്ങൾ സൃഷ്ടിക്കാൻ `@executor(id="...")` ഉപയോഗിക്കുക
   - എക്സിക്യൂട്ടറുകൾ ഡാറ്റ മാറ്റം വരുത്താനോ സൈഡ് ഇഫക്റ്റുകൾ ചെയ്യാനോ കഴിയും
   - വർക്ക്‌ഫ്ലോ ഫലങ്ങൾ നിർമ്മിക്കാൻ `ctx.yield_output()` ഉപയോഗിക്കുക

### 🚀 യാഥാർത്ഥ്യ ഉപയോഗങ്ങൾ:

- **യാത്ര ബുക്കിങ്ങ്**: ലഭ്യത പരിശോധിക്കുക, പരിഹാരങ്ങൾ നിർദ്ദേശിക്കുക, ഓപ്ഷനുകൾ താരതമ്യപ്പെടുത്തുക
- **കസ്റ്റമർ സർവീസ്**: പ്രശ്ന തരം, മനോഭാവം, മുൻഗണന പ്രകാരം റൂട്ടിംഗ്
- **ഇ-കോമേഴ്സ്**: ഇൻവെന്ററി പരിശോധിക്കുക, പരിഹാരങ്ങൾ നിർദ്ദേശിക്കുക, ഓർഡറുകൾ പ്രോസസ്സ് ചെയ്യുക
- **കോൺറന്റ് മേധാനി**: വിഷപരമായ സ്കോറുകൾ, ഉപയോക്തൃ ഫ്ലാഗുകൾ അനുസരിച്ച് റട്ട് ചെയ്യുക
- **അപ്രൂവൽ വർക്ക്‌ഫ്ലോകൾ**: തുക, ഉപയോക്തൃ പദവി, അപകടമനുഷ്യനില അനുസരിച്ച് റൂട്ടിംഗ്
- **മൾട്ടി-സ്റ്റേജ് പ്രോസസ്സിംഗ്**: ഡാറ്റയുടെ നിലവാരം, പൂർണ്ണത പ്രകാരം റൂട്ടിംഗ്

### 📚 അടുത്ത ചുവടുകൾ:

- കൂടുതൽ സങ്കീർണ്ണമായ നിബന്ധനകൾ ചേർക്കുക (ബഹുകാലിക മാനദണ്ഡങ്ങൾ)
- വർക്ക്‌ഫ്ലോ സ്റ്റേറ്റ് മാനേജുമെന്റ് ഉപയോഗിച്ച് ലൂപ്പുകൾ നടപ്പാക്കുക
- പുനരുപയോഗ സാധ്യതയുള്ള ഘടകങ്ങൾക്ക് ഉപവർക്ക്‌ഫ്ലോകൾ ചേർക്കുക
- യാഥാർത്ഥ്യ API കളുമായി സംയോജിപ്പിക്കുക (ഹോട്ടൽ ബുക്കിങ്ങ്, ഇൻവെന്ററി സിസ്റ്റങ്ങൾ)
- പിശക് കൈകാര്യം ചെയ്യലും ബാക്ക്‌അപ്പ് പാതകളും ചേർക്കുക
- ഘടനാനുസൃത ദൃശ്യവൽക്കരണ ഉപകരണങ്ങൾ ഉപയോഗിച്ച് വർക്ക്‌ഫ്ലോകൾ വീക്ഷിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
